In [21]:
from stable_baselines3 import TD3
from stable_baselines3.common.vec_env import DummyVecEnv
from stable_baselines3.common.env_util import make_vec_env
import gymnasium as gym
import os, re

In [ ]:
env = gym.make("Reacher-v5", render_mode='rgb_array')
env = make_vec_env("Reacher-v5", n_envs=8, vec_env_cls=DummyVecEnv)
model = TD3("MlpPolicy", env, 
            learning_rate=0.000_5,        # lr for all networds - Q-values, Actor, Value function
            buffer_size=1_000_000,      # replay buffer size
            learning_starts=100,        # # of data collection step before training
            batch_size=256,
            tau=0.005,                  # polyak update coefficient
            gamma=0.99,
            train_freq=1,
            gradient_steps=1, 
            action_noise=None, 
            n_steps=1,                  # n-step TD learning
            policy_delay=3,             # the policy and target networks are updated every policy_delay steps
            target_policy_noise=0.05,   # stdev of noise added to target policy
            target_noise_clip=0.1,      # limit of asbsolute value of noise
            verbose=2)
model.learn(total_timesteps=250_000)

Using cuda device
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 50       |
|    ep_rew_mean     | -76.2    |
| time/              |          |
|    episodes        | 4        |
|    fps             | 2893     |
|    time_elapsed    | 0        |
|    total_timesteps | 600      |
| train/             |          |
|    actor_loss      | 2.98     |
|    critic_loss     | 0.593    |
|    learning_rate   | 0.0005   |
|    n_updates       | 41       |
---------------------------------
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 50       |
|    ep_rew_mean     | -76.2    |
| time/              |          |
|    episodes        | 8        |
|    fps             | 2886     |
|    time_elapsed    | 0        |
|    total_timesteps | 600      |
---------------------------------
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 50       |
|    ep_rew_mean     | -76.2  

# Save the model

In [37]:
BASE_DIR = os.getcwd()
RESULT_FOLDER = 'reacher_TD3_SB_results'
RESULT_DIR = os.path.join(BASE_DIR, RESULT_FOLDER)
existing_runs = [d for d in os.listdir(RESULT_DIR) if os.path.exists(os.path.join(RESULT_DIR,d))]
run_numbers = [int(re.search(r'run_(\d{5})',d).group(1)) for d in existing_runs if re.match(r'run_\d{5}',d)]
# model.save('reacher')

trial_number = max(run_numbers, default=-1) + 1
model.save(f'{RESULT_FOLDER}/run_{trial_number:05d}')

In [ ]:
run_numbers

# Simulate the loaded model

In [40]:
model_load = TD3.load('reacher_TD3_SB_results/run_00001')

width = 1920
height = 1080
default_camera_config = {"azimuth" : 90.0, "elevation" : -90.0, "distance" : 1.5, "lookat" : [0.0, 0.0, 0.0]}

vec_env = gym.make("Reacher-v5", render_mode='human', 
                    width=width,height=height,
                    default_camera_config=default_camera_config,
                    max_episode_steps=50)

for eps in range(10):
    obs, _ = vec_env.reset()
    dones = False

    for step in range(50):
        action, _ = model_load.predict(obs, deterministic=True)
        nobs, rewards, dones, info, _ = vec_env.step(action)
        obs = nobs if not dones else vec_env.reset()
        # vec_env.render("human")

vec_env.close()

In [14]:
type(vec_env)

gymnasium.wrappers.common.TimeLimit